In [1]:
!pip uninstall -y autogen pyautogen autogen-agentchat autogen-core autogen-ext flaml ray numpy
!pip install --no-cache-dir --force-reinstall numpy==1.26.4
!pip install --no-cache-dir pyautogen==0.2.25 flaml[automl] openai


Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 321.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 202.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 227.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 348.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.7/337.7 kB 425.9 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.2.0
    Uninstalling xgboost-3.2.0:
      Successfully uninstalled xgboost-3.2.0


In [1]:
# ============================================================
# Scenario: AI Marketing Content & Validation System
# Corporate Marketing Team
# ============================================================

import os
import json
from google.colab import userdata
from openai import OpenAI

# ------------------------------------------------------------
# 1. API Setup (Groq via OpenAI-compatible client)
# ------------------------------------------------------------
groq_api_key = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = groq_api_key

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL_NAME = "llama-3.3-70b-versatile"

# ------------------------------------------------------------
# 2. Campaign Request
# ------------------------------------------------------------
campaign_request = """
Create a marketing campaign for a new AI-powered productivity app called FlowPilot.

Target audience:
- Working professionals
- Startup teams
- College students

Campaign goals:
- Increase free trial signups
- Highlight AI automation and time-saving benefits
- Maintain a modern but trustworthy tone

Required outputs:
1. Email campaign copy
2. LinkedIn post
3. Instagram caption
4. Short ad headline + ad description

Brand constraints:
- Tone should be professional, confident, and helpful
- Avoid exaggerated claims like "guaranteed success" or "instant wealth"
- Do not use fear-based manipulation
- Keep messaging aligned across all channels
- Resolve any tone mismatch between student-focused and business-focused messaging
- Standardize formatting across channels
"""

# ------------------------------------------------------------
# 3. Agent Prompts
# ------------------------------------------------------------
CONTENT_CREATOR_SYSTEM = """
You are the Content Creator Agent in a corporate marketing team.

Your task:
- Generate high-quality marketing content for multiple channels
- Keep tone professional, confident, and helpful
- Align all assets to one clear brand voice
- Handle conflicting brand guidelines by balancing business credibility with accessibility for students
- Standardize formatting across channels
- Return ONLY valid JSON

Required JSON structure:
{
  "campaign_name": "",
  "brand_voice_summary": "",
  "email": {
    "subject": "",
    "body": ""
  },
  "linkedin_post": "",
  "instagram_caption": "",
  "ad": {
    "headline": "",
    "description": ""
  },
  "visual_asset_notes": {
    "platform": "",
    "design_direction": "",
    "creative_tool_recommendation": ""
  }
}
"""

VALIDATOR_SYSTEM = """
You are the Validator Agent in a corporate marketing quality and compliance team.

Your job:
1. Review campaign content for:
   - tone consistency
   - compliance
   - brand alignment
   - formatting consistency
   - clarity across channels
2. Detect conflicting messaging between audiences or platforms
3. Simulate deployment readiness
4. Return ONLY valid JSON

Validation rules:
- No misleading claims
- No fear-based language
- Messaging must stay professional and helpful
- Content should be concise and brand-aligned
- Formatting should be channel-appropriate
- Tool/library recommendation for creative assets should be consistent and practical

Required JSON structure:
{
  "approved": true or false,
  "issues": [
    "issue 1",
    "issue 2"
  ],
  "deployment_simulation": {
    "email_ready": true or false,
    "linkedin_ready": true or false,
    "instagram_ready": true or false,
    "ad_ready": true or false
  },
  "final_recommendation": ""
}
"""

# ------------------------------------------------------------
# 4. Helper Functions
# ------------------------------------------------------------
def call_llm(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

def safe_json_load(text):
    try:
        return json.loads(text)
    except Exception:
        # fallback: try to extract JSON-like block
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1:
            return json.loads(text[start:end+1])
        raise ValueError("Model response was not valid JSON.")

def generate_campaign(campaign_brief, feedback=None):
    prompt = f"""
Campaign Brief:
{campaign_brief}

Additional validator feedback:
{feedback if feedback else "No prior feedback. Create the first draft."}

Return valid JSON only.
"""
    raw = call_llm(CONTENT_CREATOR_SYSTEM, prompt)
    return safe_json_load(raw)

def validate_campaign(campaign_json):
    prompt = f"""
Review this campaign JSON:

{json.dumps(campaign_json, indent=2)}

Return valid JSON only.
"""
    raw = call_llm(VALIDATOR_SYSTEM, prompt)
    return safe_json_load(raw)

# ------------------------------------------------------------
# 5. Multi-Agent Validation Loop
# ------------------------------------------------------------
max_iterations = 4
campaign_output = None
validation_output = None
feedback = None

for iteration in range(1, max_iterations + 1):
    print(f"\n{'='*70}")
    print(f"ITERATION {iteration}")
    print(f"{'='*70}")

    campaign_output = generate_campaign(campaign_request, feedback)
    print("Content Creator Agent completed draft.")

    validation_output = validate_campaign(campaign_output)
    print("Validator Agent reviewed draft.")

    print("\nValidation Summary:")
    print(json.dumps(validation_output, indent=2))

    if validation_output.get("approved") is True:
        print("\nCampaign approved.")
        break
    else:
        issues = validation_output.get("issues", [])
        feedback = "Please fix the following issues:\n- " + "\n- ".join(issues)
        print("\nCampaign not approved. Sending feedback back to Content Creator Agent.")

# ------------------------------------------------------------
# 6. Save Final Approved Assets
# ------------------------------------------------------------
final_output = {
    "campaign_request": campaign_request.strip(),
    "approved_campaign_assets": campaign_output,
    "validation_report": validation_output
}

with open("approved_campaign_assets.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

with open("approved_campaign_assets.txt", "w", encoding="utf-8") as f:
    f.write("AI Marketing Content & Validation System\n")
    f.write("=" * 60 + "\n\n")
    f.write("CAMPAIGN REQUEST\n")
    f.write("-" * 60 + "\n")
    f.write(campaign_request.strip() + "\n\n")

    f.write("FINAL APPROVED CAMPAIGN ASSETS\n")
    f.write("-" * 60 + "\n")
    f.write(json.dumps(campaign_output, indent=2, ensure_ascii=False) + "\n\n")

    f.write("VALIDATION REPORT\n")
    f.write("-" * 60 + "\n")
    f.write(json.dumps(validation_output, indent=2, ensure_ascii=False) + "\n")

print("\nFiles saved successfully:")
print("- approved_campaign_assets.json")
print("- approved_campaign_assets.txt")

# ------------------------------------------------------------
# 7. Final Console Display
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL CAMPAIGN ASSETS")
print("=" * 70)
print(json.dumps(campaign_output, indent=2, ensure_ascii=False))

print("\n" + "=" * 70)
print("FINAL VALIDATION REPORT")
print("=" * 70)
print(json.dumps(validation_output, indent=2, ensure_ascii=False))

print("\nDONE")


ITERATION 1
Content Creator Agent completed draft.
Validator Agent reviewed draft.

Validation Summary:
{
  "approved": true,
  "issues": [],
  "deployment_simulation": {
    "email_ready": true,
    "linkedin_ready": true,
    "instagram_ready": true,
    "ad_ready": true
  },
  "final_recommendation": "The campaign content is well-aligned with the brand voice and tone. The messaging is consistent, professional, and helpful across all channels. The visual asset notes provide clear direction for creative assets. The campaign is ready for deployment."
}

Campaign approved.

Files saved successfully:
- approved_campaign_assets.json
- approved_campaign_assets.txt

FINAL CAMPAIGN ASSETS
{
  "campaign_name": "FlowPilot Productivity Boost",
  "brand_voice_summary": "Empowering individuals and teams to achieve more with AI-driven productivity solutions, while maintaining a professional, confident, and helpful tone.",
  "email": {
    "subject": "Unlock Your Full Potential with FlowPilot",
  